In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
from src.models.model_loader import (
    load_vectorizer
)

vectorizer = load_vectorizer()

df = pd.read_csv("../data/processed/cleaned_jobs.csv")

In [3]:
X = vectorizer.transform(
    df["combined_text"]
)

y = df["fraudulent"]

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
!pip install optuna

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: C:\Program Files\Python312\python.exe -m pip install --upgrade pip


In [7]:
import optuna

from sklearn.metrics import f1_score

from xgboost import XGBClassifier

In [8]:
import sys

print(sys.executable)

d:\DIVY\CODING\PURECODING\AI-ML\Week9\Devops\JobClarity\.venv\Scripts\python.exe


In [9]:
import optuna

print(optuna.__version__)

4.9.0


In [11]:
def objective(trial):

    model = XGBClassifier(

        n_estimators=trial.suggest_int(
            "n_estimators",
            100,
            500
        ),

        max_depth=trial.suggest_int(
            "max_depth",
            3,
            10
        ),

        learning_rate=trial.suggest_float(
            "learning_rate",
            0.01,
            0.3
        ),

        subsample=trial.suggest_float(
            "subsample",
            0.6,
            1.0
        ),

        colsample_bytree=trial.suggest_float(
            "colsample_bytree",
            0.6,
            1.0
        ),

        eval_metric="logloss",

        random_state=42
    )

    model.fit(X_train, y_train)

    pred = model.predict(X_test)

    score = f1_score(y_test, pred)

    return score

In [12]:
study = optuna.create_study(
    direction="maximize"
)

study.optimize(
    objective,
    n_trials=20,
    show_progress_bar=True
)

[I 2026-07-31 00:41:36,044] A new study created in memory with name: no-name-2b5fd3bc-e639-4a4e-b221-688f50ad90b9


  0%|          | 0/20 [00:00<?, ?it/s]

[I 2026-07-31 00:42:39,706] Trial 0 finished with value: 0.7526881720430108 and parameters: {'n_estimators': 161, 'max_depth': 5, 'learning_rate': 0.053988363044446645, 'subsample': 0.7523766971152486, 'colsample_bytree': 0.8205926272872204}. Best is trial 0 with value: 0.7526881720430108.
[I 2026-07-31 00:46:02,041] Trial 1 finished with value: 0.7418181818181818 and parameters: {'n_estimators': 440, 'max_depth': 6, 'learning_rate': 0.014167033464045431, 'subsample': 0.6139038474351984, 'colsample_bytree': 0.8197461935707986}. Best is trial 0 with value: 0.7526881720430108.
[I 2026-07-31 00:47:12,164] Trial 2 finished with value: 0.822742474916388 and parameters: {'n_estimators': 378, 'max_depth': 3, 'learning_rate': 0.15655853919834686, 'subsample': 0.8054602086379644, 'colsample_bytree': 0.6787482347196301}. Best is trial 2 with value: 0.822742474916388.
[I 2026-07-31 00:49:00,317] Trial 3 finished with value: 0.7903780068728522 and parameters: {'n_estimators': 206, 'max_depth': 7, 

In [13]:
from xgboost import XGBClassifier

best_model = XGBClassifier(
    n_estimators=359,
    max_depth=8,
    learning_rate=0.29551139764656026,
    subsample=0.8890683880722694,
    colsample_bytree=0.75118868399315,
    eval_metric="logloss",
    random_state=42
)

best_model.fit(
    X_train,
    y_train
)

,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.75118868399315
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'logloss'
,feature_types feature_types: typing.Optional[typing.Sequence[str]].. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [14]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

pred = best_model.predict(X_test)

prob = best_model.predict_proba(X_test)[:,1]

print("Accuracy :", accuracy_score(y_test,pred))
print("Precision:", precision_score(y_test,pred))
print("Recall   :", recall_score(y_test,pred))
print("F1       :", f1_score(y_test,pred))
print("ROC AUC  :", roc_auc_score(y_test,prob))

Accuracy : 0.9854586129753915
Precision: 0.9618320610687023
Recall   : 0.7283236994219653
F1       : 0.8289473684210527
ROC AUC  : 0.9835337402054292


In [15]:
import joblib

joblib.dump(
    best_model,
    "../models/xgboost_model_tuned.pkl"
)

print("Saved Successfully")

Saved Successfully


In [16]:
baseline_metrics = {
    "Accuracy": 0.9815,
    "Precision": 0.9735,
    "Recall": 0.6358,
    "F1 Score": 0.7692,
    "ROC AUC": 0.9887
}

tuned_metrics = {
    "Accuracy": accuracy_score(y_test, pred),
    "Precision": precision_score(y_test, pred),
    "Recall": recall_score(y_test, pred),
    "F1 Score": f1_score(y_test, pred),
    "ROC AUC": roc_auc_score(y_test, prob)
}

comparison_df = pd.DataFrame({
    "Metric": baseline_metrics.keys(),
    "Baseline": baseline_metrics.values(),
    "Tuned": tuned_metrics.values()
})

comparison_df["Improvement"] = comparison_df["Tuned"] - comparison_df["Baseline"]

comparison_df

,Metric,Baseline,Tuned,Improvement
0,Accuracy,0.9815,0.985459,0.003959
1,Precision,0.9735,0.961832,-0.011668
2,Recall,0.6358,0.728324,0.092524
3,F1 Score,0.7692,0.828947,0.059747
4,ROC AUC,0.9887,0.983534,-0.005166


In [17]:
comparison_df.to_csv(
    "../artifacts/model_comparison.csv",
    index=False
)

print("✅ Model comparison saved.")

✅ Model comparison saved.


In [18]:
print("Selected Production Model: Tuned XGBoost")

print("\nReason:")

print("- Higher Recall")
print("- Higher F1 Score")
print("- Better fraud detection")
print("- Suitable for production")

Selected Production Model: Tuned XGBoost

Reason:
- Higher Recall
- Higher F1 Score
- Better fraud detection
- Suitable for production


In [19]:
from src.models.model_loader import load_model

model = load_model()

print(type(model))

<class 'xgboost.sklearn.XGBClassifier'>
